# CHORUS Research Design Atlas

This atlas treats CHORUS as a transparent synthetic laboratory. It defines research questions that can be tested **within variants of the model**, while keeping any claim about real people or institutions contingent on separate empirical work.

The goal is not to maximize the number of simulated runs. The goal is to connect each question to a mechanism, comparison, observation plan, interpretation rule, and stopping condition.

In [1]:
from html import escape
from hashlib import sha256
import re

class HTMLResult(str):
    def _repr_html_(self):
        return str(self)

def table_html(caption, columns, rows, row_headers=False):
    head = "".join(f'<th scope="col">{escape(str(column))}</th>' for column in columns)
    body_rows = []
    for row in rows:
        rendered = []
        for index, value in enumerate(row):
            tag = "th" if row_headers and index == 0 else "td"
            scope = ' scope="row"' if tag == "th" else ""
            rendered.append(f'<{tag}{scope}>{escape(str(value))}</{tag}>')
        body_rows.append("<tr>" + "".join(rendered) + "</tr>")
    label = escape(caption)
    return HTMLResult(
        f'<div class="table-wrap" role="region" aria-label="{label}" tabindex="0">'
        f'<table><caption>{label}</caption><thead><tr>{head}</tr></thead>'
        f'<tbody>{"".join(body_rows)}</tbody></table></div>'
    )

def cards_html(title, cards):
    items = []
    for label, value, note in cards:
        items.append(
            '<article class="metric-card">'
            f'<h4>{escape(str(label))}</h4><strong>{escape(str(value))}</strong>'
            f'<p>{escape(str(note))}</p></article>'
        )
    return HTMLResult(f'<section class="metric-grid" aria-label="{escape(title)}">{"".join(items)}</section>')

def checklist_html(title, rows):
    items = []
    for status, label, evidence in rows:
        items.append(
            '<li>'
            f'<span class="status">{escape(status)}</span>'
            f'<strong>{escape(label)}</strong><p>{escape(evidence)}</p>'
            '</li>'
        )
    return HTMLResult(f'<section class="checklist" aria-label="{escape(title)}"><ul>{"".join(items)}</ul></section>')

DIAGRAM_STYLE = r"""
.diagram-figure{margin:1rem 0;padding:1rem;border:1px solid rgba(213,222,220,.24);border-radius:.8rem 1.3rem .9rem 1.1rem;background:rgba(4,16,10,.76)}
.diagram-figure figcaption{display:grid;gap:.3rem;margin-bottom:.8rem}.diagram-figure figcaption span{color:#e2c57f;font:700 .72rem/1.35 ui-monospace,SFMono-Regular,Consolas,monospace;letter-spacing:.08em;text-transform:uppercase}.diagram-figure figcaption strong{font-size:1.25rem;color:#f2f1e8}.diagram-figure figcaption p{max-width:78ch;margin:0;color:#c7d1c9}
.diagram-canvas{max-width:100%;overflow-x:auto;border:1px solid rgba(213,222,220,.16);border-radius:.65rem;background:#06130c}.diagram-svg{display:block;width:100%;min-width:760px;height:auto}.diagram-legend{display:flex;flex-wrap:wrap;gap:.5rem 1rem;margin:.8rem 0 0;padding:0;list-style:none;color:#c7d1c9;font-size:.8rem}.diagram-legend li{display:flex;align-items:center;gap:.4rem}.diagram-key{width:1.8rem;height:.2rem;display:inline-block;background:#e2c57f}.diagram-key-data{background:#a7e0bd}.diagram-key-evidence{height:0;border-top:2px dashed #c9b6db;background:none}.diagram-key-boundary{height:0;border-top:2px dashed #d5dedc;background:none}.diagram-key-association{background:#b8c0bc}.diagram-assurance{display:block;margin-top:.7rem;color:#a7e0bd;font:700 .72rem/1.4 ui-monospace,SFMono-Regular,Consolas,monospace}.diagram-equivalent{margin-top:.7rem;border-top:1px solid rgba(213,222,220,.18)}.diagram-equivalent summary{min-height:44px;padding:.7rem 0;cursor:pointer;color:#d5dedc}.diagram-equivalent h4{margin:.7rem 0 .3rem;color:#e2c57f}.diagram-equivalent ul{margin:.2rem 0 0;padding-left:1.2rem}.diagram-notes{color:#c7d1c9}
@media(max-width:42rem){.diagram-figure{padding:.65rem}.diagram-svg{min-width:700px}}
@media(forced-colors:active){.diagram-figure,.diagram-canvas{border:1px solid CanvasText}}
"""

SVG_COLORS = {
    "canvas": "#06130c",
    "group_fill": "#0b2819",
    "group_stroke": "#799886",
    "person_fill": "#163e2a",
    "person_stroke": "#a7e0bd",
    "system_fill": "#103522",
    "system_stroke": "#e2c57f",
    "component_fill": "#0d281a",
    "component_stroke": "#9bc8aa",
    "data_fill": "#262b22",
    "data_stroke": "#d5dedc",
    "evidence_fill": "#282433",
    "evidence_stroke": "#c9b6db",
    "boundary_fill": "#2d271b",
    "boundary_stroke": "#e2c57f",
    "risk_fill": "#351f1f",
    "risk_stroke": "#e0a8a8",
    "decision_fill": "#352b18",
    "decision_stroke": "#e2c57f",
    "title": "#f2f1e8",
    "body": "#c7d1c9",
    "role": "#a7e0bd",
    "flow": "#e2c57f",
    "data": "#a7e0bd",
    "evidence": "#c9b6db",
    "boundary": "#d5dedc",
    "association": "#b8c0bc",
    "label_bg": "#07140d",
    "label_stroke": "#5a6c60",
}


def dnode(node_id, x, y, w, h, title, body="", kind="component", shape="rect", role=""):
    return {
        "id": node_id, "x": float(x), "y": float(y), "w": float(w), "h": float(h),
        "title": title, "body": body, "kind": kind, "shape": shape, "role": role,
    }


def dedge(source, target, points, kind="flow", label="", label_at=None, arrow=True):
    return {
        "source": source, "target": target,
        "points": tuple((float(x), float(y)) for x, y in points),
        "kind": kind, "label": label, "label_at": label_at, "arrow": arrow,
    }


def dgroup(group_id, x, y, w, h, label, kind="boundary"):
    return {
        "id": group_id, "x": float(x), "y": float(y), "w": float(w),
        "h": float(h), "label": label, "kind": kind,
    }


def _slug(value):
    cleaned = re.sub(r"[^a-z0-9]+", "-", value.lower()).strip("-")
    return cleaned or "diagram"


def _lines(value, max_chars, max_lines=4):
    raw_lines = str(value).split("\n") if value else []
    lines = []
    for raw in raw_lines:
        words = raw.split()
        if not words:
            lines.append("")
            continue
        current = words[0]
        for word in words[1:]:
            candidate = current + " " + word
            if len(candidate) <= max_chars:
                current = candidate
            else:
                lines.append(current)
                current = word
        lines.append(current)
    if len(lines) > max_lines:
        lines = lines[:max_lines]
        lines[-1] = lines[-1].rstrip(" …") + "…"
    return lines


def _segments(points):
    return list(zip(points, points[1:]))


def _on_boundary(point, node, tolerance=0.01):
    x, y = point
    left, top = node["x"], node["y"]
    right, bottom = left + node["w"], top + node["h"]
    on_vertical = (
        (abs(x - left) <= tolerance or abs(x - right) <= tolerance)
        and top - tolerance <= y <= bottom + tolerance
    )
    on_horizontal = (
        (abs(y - top) <= tolerance or abs(y - bottom) <= tolerance)
        and left - tolerance <= x <= right + tolerance
    )
    return on_vertical or on_horizontal


def _segment_axis(segment):
    (x1, y1), (x2, y2) = segment
    if x1 == x2 and y1 != y2:
        return "v"
    if y1 == y2 and x1 != x2:
        return "h"
    raise AssertionError(f"Diagram route segment must be orthogonal and nonzero: {segment}")


def _segment_crosses_rect(segment, node):
    (x1, y1), (x2, y2) = segment
    left, top = node["x"], node["y"]
    right, bottom = left + node["w"], top + node["h"]
    axis = _segment_axis(segment)
    if axis == "h":
        if not top < y1 < bottom:
            return False
        return max(min(x1, x2), left) < min(max(x1, x2), right)
    if not left < x1 < right:
        return False
    return max(min(y1, y2), top) < min(max(y1, y2), bottom)


def _segment_intersection(first, second):
    a1, a2 = first
    b1, b2 = second
    axis_a, axis_b = _segment_axis(first), _segment_axis(second)
    if axis_a != axis_b:
        horizontal = first if axis_a == "h" else second
        vertical = second if axis_a == "h" else first
        (hx1, hy), (hx2, _) = horizontal
        (vx, vy1), (_, vy2) = vertical
        if min(hx1, hx2) <= vx <= max(hx1, hx2) and min(vy1, vy2) <= hy <= max(vy1, vy2):
            return (vx, hy)
        return None
    if axis_a == "h" and a1[1] == b1[1]:
        lo = max(min(a1[0], a2[0]), min(b1[0], b2[0]))
        hi = min(max(a1[0], a2[0]), max(b1[0], b2[0]))
        if lo < hi:
            return ("overlap", lo, hi, a1[1])
        if lo == hi:
            return (lo, a1[1])
    if axis_a == "v" and a1[0] == b1[0]:
        lo = max(min(a1[1], a2[1]), min(b1[1], b2[1]))
        hi = min(max(a1[1], a2[1]), max(b1[1], b2[1]))
        if lo < hi:
            return ("overlap", lo, hi, a1[0])
        if lo == hi:
            return (a1[0], lo)
    return None


def _rectangles_overlap(first, second):
    return (
        max(first["x"], second["x"]) < min(first["x"] + first["w"], second["x"] + second["w"])
        and max(first["y"], second["y"]) < min(first["y"] + first["h"], second["y"] + second["h"])
    )


def _validate_diagram(width, height, nodes, edges):
    assert width > 0 and height > 0
    node_map = {node["id"]: node for node in nodes}
    assert len(node_map) == len(nodes), "Diagram node IDs must be unique."
    for node in nodes:
        assert node["w"] > 0 and node["h"] > 0
        assert 0 <= node["x"] < width and 0 <= node["y"] < height
        assert node["x"] + node["w"] <= width and node["y"] + node["h"] <= height
    for index, first in enumerate(nodes):
        for second in nodes[index + 1:]:
            assert not _rectangles_overlap(first, second), (
                f"Diagram nodes overlap: {first['id']} and {second['id']}"
            )

    all_segments = []
    for edge_index, edge in enumerate(edges):
        assert edge["source"] in node_map and edge["target"] in node_map
        points = edge["points"]
        assert len(points) >= 2
        assert _on_boundary(points[0], node_map[edge["source"]]), (
            f"Route must start on source boundary: {edge}"
        )
        assert _on_boundary(points[-1], node_map[edge["target"]]), (
            f"Route must end on target boundary: {edge}"
        )
        for segment_index, segment in enumerate(_segments(points)):
            _segment_axis(segment)
            for node_id, node in node_map.items():
                if node_id in (edge["source"], edge["target"]):
                    continue
                assert not _segment_crosses_rect(segment, node), (
                    f"Route crosses node {node_id}: {edge}"
                )
            all_segments.append((edge_index, segment_index, edge, segment))

    for index, first in enumerate(all_segments):
        for second in all_segments[index + 1:]:
            edge_a, edge_b = first[2], second[2]
            if first[0] == second[0]:
                continue
            intersection = _segment_intersection(first[3], second[3])
            if intersection is None:
                continue
            shared_terminal_points = (
                set((edge_a["points"][0], edge_a["points"][-1]))
                & set((edge_b["points"][0], edge_b["points"][-1]))
            )
            if (
                isinstance(intersection, tuple)
                and intersection
                and intersection[0] != "overlap"
                and intersection in shared_terminal_points
            ):
                continue
            raise AssertionError(
                f"Diagram routes cross or overlap at {intersection}: {edge_a} / {edge_b}"
            )
    return {
        "nodes": len(nodes), "edges": len(edges), "segments": len(all_segments),
        "crossings": 0, "node_incursions": 0, "node_overlaps": 0,
    }


def _svg_text(x, y, lines, fill, size, weight=400, line_height=15, anchor="start", letter_spacing=0):
    if not lines:
        return ""
    spans = []
    for index, line in enumerate(lines):
        dy = 0 if index == 0 else line_height
        spans.append(f'<tspan x="{x:g}" dy="{dy:g}">{escape(line)}</tspan>')
    return (
        f'<text x="{x:g}" y="{y:g}" fill="{fill}" font-size="{size:g}" '
        f'font-weight="{weight}" text-anchor="{anchor}" letter-spacing="{letter_spacing:g}" '
        'font-family="Inter, ui-sans-serif, -apple-system, BlinkMacSystemFont, Segoe UI, sans-serif">'
        f'{"".join(spans)}</text>'
    )


def _node_colors(kind):
    return (
        SVG_COLORS.get(f"{kind}_fill", SVG_COLORS["component_fill"]),
        SVG_COLORS.get(f"{kind}_stroke", SVG_COLORS["component_stroke"]),
    )


def _edge_dash(kind):
    if kind == "evidence":
        return ' stroke-dasharray="7 5"'
    if kind == "boundary":
        return ' stroke-dasharray="3 5"'
    return ""


def _figure_shell(diagram_type, title, description, svg, assurance, legend=(), notes=(), equivalent=""):
    legend_html = ""
    if legend:
        legend_items = ''.join(
            f'<li><i class="diagram-key diagram-key-{escape(kind)}" aria-hidden="true"></i>'
            f'<span>{escape(label)}</span></li>'
            for kind, label in legend
        )
        legend_html = f'<ul class="diagram-legend" aria-label="Diagram legend">{legend_items}</ul>'
    notes_html = ''.join(f'<li>{escape(str(note))}</li>' for note in notes)
    if notes_html:
        notes_html = f'<ul class="diagram-notes">{notes_html}</ul>'
    return HTMLResult(
        f'<figure class="diagram-figure" data-diagram-type="{escape(diagram_type)}" '
        'data-routing="orthogonal-crossing-free">'
        f'<figcaption><span>{escape(diagram_type)}</span><strong>{escape(title)}</strong>'
        f'<p>{escape(description)}</p></figcaption>'
        f'<div class="diagram-canvas" role="region" aria-label="{escape(title)} diagram" tabindex="0">{svg}</div>'
        f'{legend_html}{notes_html}<span class="diagram-assurance">{escape(assurance)}</span>{equivalent}'
        '</figure>'
    )


def diagram_html(diagram_type, title, description, width, height, nodes, edges, groups=(), legend=(), notes=()):
    nodes = tuple(nodes)
    edges = tuple(edges)
    groups = tuple(groups)
    assurance = _validate_diagram(width, height, nodes, edges)
    uid = _slug(title) + "-" + sha256(title.encode("utf-8")).hexdigest()[:8]
    title_id, desc_id = uid + "-title", uid + "-desc"
    node_map = {node["id"]: node for node in nodes}

    marker_kinds = sorted(set(edge.get("kind", "flow") for edge in edges if edge.get("arrow", True)))
    defs = []
    for kind in marker_kinds:
        marker = uid + "-arrow-" + _slug(kind)
        color = SVG_COLORS.get(kind, SVG_COLORS["flow"])
        defs.append(
            f'<marker id="{marker}" viewBox="0 0 10 10" refX="9" refY="5" '
            'markerWidth="7" markerHeight="7" orient="auto-start-reverse">'
            f'<path d="M 0 0 L 10 5 L 0 10 z" fill="{color}"/></marker>'
        )

    parts = [
        f'<svg class="diagram-svg" width="{width:g}" height="{height:g}" '
        f'viewBox="0 0 {width:g} {height:g}" role="img" aria-labelledby="{title_id} {desc_id}">',
        f'<title id="{title_id}">{escape(title)}</title>',
        f'<desc id="{desc_id}">{escape(description)}</desc>',
        f'<rect x="0" y="0" width="{width:g}" height="{height:g}" fill="{SVG_COLORS["canvas"]}"/>',
        f'<defs>{"".join(defs)}</defs>',
    ]

    for group in groups:
        parts.append(
            f'<rect x="{group["x"]:g}" y="{group["y"]:g}" width="{group["w"]:g}" '
            f'height="{group["h"]:g}" rx="18" fill="{SVG_COLORS["group_fill"]}" fill-opacity=".28" '
            f'stroke="{SVG_COLORS["group_stroke"]}" stroke-width="1.2" stroke-dasharray="7 5"/>'
        )
        parts.append(
            _svg_text(group["x"] + 14, group["y"] + 21, [group["label"]], SVG_COLORS["title"], 12, 700, 14, "start", .7)
        )

    for edge in edges:
        points = " ".join(f'{x:g},{y:g}' for x, y in edge["points"])
        color = SVG_COLORS.get(edge["kind"], SVG_COLORS["flow"])
        marker_attr = ""
        if edge.get("arrow", True):
            marker_attr = f' marker-end="url(#{uid}-arrow-{_slug(edge["kind"])})"'
        parts.append(
            f'<polyline points="{points}" fill="none" stroke="{color}" stroke-width="2" '
            f'stroke-linecap="round" stroke-linejoin="round"{_edge_dash(edge["kind"])}{marker_attr}/>'
        )
        if edge.get("label"):
            lx, ly = edge.get("label_at") or edge["points"][len(edge["points"]) // 2]
            label_width = max(72, min(150, len(edge["label"]) * 6.4 + 18))
            parts.append(
                f'<rect x="{lx - label_width / 2:g}" y="{ly - 12:g}" width="{label_width:g}" height="22" '
                f'rx="7" fill="{SVG_COLORS["label_bg"]}" stroke="{SVG_COLORS["label_stroke"]}" stroke-width=".8"/>'
            )
            parts.append(_svg_text(lx, ly + 3, [edge["label"]], SVG_COLORS["title"], 10, 700, 12, "middle"))

    for node in nodes:
        x, y, w, h = node["x"], node["y"], node["w"], node["h"]
        shape = node.get("shape", "rect")
        fill, stroke = _node_colors(node.get("kind", "component"))
        common = f'fill="{fill}" stroke="{stroke}" stroke-width="1.6"'
        if shape == "diamond":
            points = f'{x + w / 2:g},{y:g} {x + w:g},{y + h / 2:g} {x + w / 2:g},{y + h:g} {x:g},{y + h / 2:g}'
            parts.append(f'<polygon points="{points}" {common}/>' )
        elif shape == "pill":
            parts.append(f'<rect x="{x:g}" y="{y:g}" width="{w:g}" height="{h:g}" rx="{h / 2:g}" {common}/>' )
        elif shape == "document":
            fold = min(18, w * .12)
            d = (
                f'M {x:g} {y:g} H {x + w - fold:g} L {x + w:g} {y + fold:g} '
                f'V {y + h:g} H {x:g} Z M {x + w - fold:g} {y:g} V {y + fold:g} H {x + w:g}'
            )
            parts.append(f'<path d="{d}" {common} stroke-linejoin="round"/>' )
        else:
            parts.append(f'<rect x="{x:g}" y="{y:g}" width="{w:g}" height="{h:g}" rx="12" {common}/>' )

        char_width = max(13, int((w - 24) / 7.1))
        title_lines = _lines(node["title"], char_width, 2)
        body_lines = _lines(node.get("body", ""), char_width, 4)
        role = node.get("role", "")
        top = y + 20
        if role:
            parts.append(_svg_text(x + w / 2, top, [role.upper()], SVG_COLORS["role"], 10, 700, 12, "middle", .8))
            top += 18
        parts.append(_svg_text(x + w / 2, top, title_lines, SVG_COLORS["title"], 14, 700, 16, "middle"))
        body_y = top + 16 * len(title_lines) + 5
        parts.append(_svg_text(x + w / 2, body_y, body_lines, SVG_COLORS["body"], 11.5, 400, 14, "middle"))

    parts.append('</svg>')
    relation_items = []
    for edge in edges:
        relation = f'{node_map[edge["source"]]["title"]} → {node_map[edge["target"]]["title"]}'
        if edge.get("label"):
            relation += f' ({edge["label"]})'
        relation_items.append(f'<li>{escape(relation)}</li>')
    node_items = [
        f'<li><strong>{escape(node["title"])}</strong>'
        f'{": " + escape(node["body"]) if node.get("body") else ""}</li>'
        for node in nodes
    ]
    equivalent = (
        '<details class="diagram-equivalent"><summary>Text equivalent</summary>'
        f'<h4>Elements</h4><ul>{"".join(node_items)}</ul>'
        f'<h4>Relationships</h4><ul>{"".join(relation_items) if relation_items else "<li>No connector relationships; the diagram uses nested evidentiary zones.</li>"}</ul>'
        '</details>'
    )
    assurance_text = (
        f'Validated: {assurance["nodes"]} nodes · {assurance["edges"]} edges · '
        'orthogonal routing · 0 crossings · 0 node incursions · 0 node overlaps'
    )
    return _figure_shell(
        diagram_type, title, description, ''.join(parts), assurance_text,
        legend=legend, notes=notes, equivalent=equivalent,
    )


def matrix_diagram_html(diagram_type, title, description, rows, columns, coverage, notes=()):
    rows = tuple(rows)
    columns = tuple(columns)
    valid_marks = {"P", "S", ""}
    assert len(set(rows)) == len(rows) and len(set(columns)) == len(columns)
    for key, mark in coverage.items():
        assert key[0] in rows and key[1] in columns and mark in valid_marks

    left = 255
    top = 125
    cell_w = 125
    cell_h = 72
    right_pad = 25
    bottom_pad = 35
    width = left + cell_w * len(columns) + right_pad
    height = top + cell_h * len(rows) + bottom_pad
    uid = _slug(title) + "-" + sha256(title.encode("utf-8")).hexdigest()[:8]
    title_id, desc_id = uid + "-title", uid + "-desc"
    parts = [
        f'<svg class="diagram-svg" width="{width:g}" height="{height:g}" viewBox="0 0 {width:g} {height:g}" '
        f'role="img" aria-labelledby="{title_id} {desc_id}">',
        f'<title id="{title_id}">{escape(title)}</title>',
        f'<desc id="{desc_id}">{escape(description)}</desc>',
        f'<rect x="0" y="0" width="{width:g}" height="{height:g}" fill="{SVG_COLORS["canvas"]}"/>',
    ]
    for column_index, column in enumerate(columns):
        x = left + column_index * cell_w
        lines = _lines(column, 15, 3)
        parts.append(_svg_text(x + cell_w / 2, 38, lines, SVG_COLORS["title"], 11, 700, 14, "middle"))
    for row_index, row in enumerate(rows):
        y = top + row_index * cell_h
        parts.append(
            f'<rect x="8" y="{y:g}" width="{left - 16:g}" height="{cell_h:g}" rx="8" '
            f'fill="{SVG_COLORS["component_fill"]}" stroke="{SVG_COLORS["component_stroke"]}" stroke-width="1"/>'
        )
        parts.append(_svg_text(20, y + 28, _lines(row, 30, 2), SVG_COLORS["title"], 12, 700, 15, "start"))
        for column_index, column in enumerate(columns):
            x = left + column_index * cell_w
            mark = coverage.get((row, column), "")
            if mark == "P":
                fill, stroke, label = "#173d29", "#a7e0bd", "P"
            elif mark == "S":
                fill, stroke, label = "#2b2835", "#c9b6db", "S"
            else:
                fill, stroke, label = "#0a1a11", "#38483e", "—"
            parts.append(
                f'<rect x="{x:g}" y="{y:g}" width="{cell_w:g}" height="{cell_h:g}" '
                f'fill="{fill}" stroke="{stroke}" stroke-width="1"/>'
            )
            parts.append(_svg_text(x + cell_w / 2, y + 42, [label], SVG_COLORS["title"] if mark else SVG_COLORS["body"], 17, 700, 18, "middle"))
    parts.append('</svg>')

    table_rows = []
    for row in rows:
        table_rows.append((row, *({"P": "Primary", "S": "Supporting", "": "Not claimed"}[coverage.get((row, column), "")] for column in columns)))
    equivalent = str(table_html(
        title + " text equivalent",
        ("Test family", *columns),
        table_rows,
        row_headers=True,
    ))
    equivalent = f'<details class="diagram-equivalent"><summary>Text equivalent</summary>{equivalent}</details>'
    return _figure_shell(
        diagram_type, title, description, ''.join(parts),
        f'Validated: {len(rows)} test families · {len(columns)} concern columns · matrix topology · 0 connector lines',
        legend=(("data", "P = primary coverage"), ("evidence", "S = supporting coverage")),
        notes=notes,
        equivalent=equivalent,
    )


_html = HTMLResult(f"<style>{DIAGRAM_STYLE}</style>")
print("Standard-library rendering helpers loaded, including deterministic SVG diagrams with crossing validation.")
_html

Standard-library rendering helpers loaded, including deterministic SVG diagrams with crossing validation.


.diagram-figure{margin:1rem 0;padding:1rem;border:1px solid rgba(213,222,220,.24);border-radius:.8rem 1.3rem .9rem 1.1rem;background:rgba(4,16,10,.76)}
.diagram-figure figcaption{display:grid;gap:.3rem;margin-bottom:.8rem}.diagram-figure figcaption span{color:#e2c57f;font:700 .72rem/1.35 ui-monospace,SFMono-Regular,Consolas,monospace;letter-spacing:.08em;text-transform:uppercase}.diagram-figure figcaption strong{font-size:1.25rem;color:#f2f1e8}.diagram-figure figcaption p{max-width:78ch;margin:0;color:#c7d1c9}
.diagram-canvas{max-width:100%;overflow-x:auto;border:1px solid rgba(213,222,220,.16);border-radius:.65rem;background:#06130c}.diagram-svg{display:block;width:100%;min-width:760px;height:auto}.diagram-legend{display:flex;flex-wrap:wrap;gap:.5rem 1rem;margin:.8rem 0 0;padding:0;list-style:none;color:#c7d1c9;font-size:.8rem}.diagram-legend li{display:flex;align-items:center;gap:.4rem}.diagram-key{width:1.8rem;height:.2rem;display:inline-block;background:#e2c57f}.diagram-key-data{background:#a7e0bd}.diagram-key-evidence{height:0;border-top:2px dashed #c9b6db;background:none}.diagram-key-boundary{height:0;border-top:2px dashed #d5dedc;background:none}.diagram-key-association{background:#b8c0bc}.diagram-assurance{display:block;margin-top:.7rem;color:#a7e0bd;font:700 .72rem/1.4 ui-monospace,SFMono-Regular,Consolas,monospace}.diagram-equivalent{margin-top:.7rem;border-top:1px solid rgba(213,222,220,.18)}.diagram-equivalent summary{min-height:44px;padding:.7rem 0;cursor:pointer;color:#d5dedc}.diagram-equivalent h4{margin:.7rem 0 .3rem;color:#e2c57f}.diagram-equivalent ul{margin:.2rem 0 0;padding-left:1.2rem}.diagram-notes{color:#c7d1c9}
@media(max-width:42rem){.diagram-figure{padding:.65rem}.diagram-svg{min-width:700px}}
@media(forced-colors:active){.diagram-figure,.diagram-canvas{border:1px solid CanvasText}}

## Research posture

A CHORUS study begins with a bounded question and a declared contrast. It uses deterministic seeds to preserve matched worlds, event receipts to explain differences, and qualitative review to determine whether a numerically valid run remains socially and narratively coherent.

In [2]:
principles = [
    ("Question before metric", "Choose the mechanism and comparison before selecting an outcome index."),
    ("Matched worlds", "Use the same seed and policy history when changing one model condition."),
    ("Mechanism trace", "Retain the event and effect receipts that connect condition to outcome."),
    ("Finite-domain honesty", "Report the exact seed set, policy set, exclusions, and failures."),
    ("Mixed evidence", "Combine invariant checks and distributions with close reading of actor, relationship, affect, and narrative continuity."),
    ("No synthetic generalization", "Do not translate generated frequency into human prevalence or prediction."),
    ("Revision is a result", "Treat incoherence, instability, or sensitivity as evidence that the model or claim needs narrowing."),
]
_html = checklist_html("Research-design principles", [("REQUIRED", label, note) for label, note in principles])
print("Seven principles govern every proposed CHORUS study.")
_html

Seven principles govern every proposed CHORUS study.


REQUIRED  Question before metric  Choose the mechanism and comparison before selecting an outcome index.    REQUIRED  Matched worlds  Use the same seed and policy history when changing one model condition.    REQUIRED  Mechanism trace  Retain the event and effect receipts that connect condition to outcome.    REQUIRED  Finite-domain honesty  Report the exact seed set, policy set, exclusions, and failures.    REQUIRED  Mixed evidence  Combine invariant checks and distributions with close reading of actor, relationship, affect, and narrative continuity.    REQUIRED  No synthetic generalization  Do not translate generated frequency into human prevalence or prediction.    REQUIRED  Revision is a result  Treat incoherence, instability, or sensitivity as evidence that the model or claim needs narrowing.

### Research design map

A valid study preserves the chain from construct to operationalization to measure to output. The parallel rows prevent one convenient metric from silently substituting for several distinct research questions.

In [3]:
cols=[('c','Construct'),('o','Operationalization'),('m','Measure'),('x','Output')]
xs=[30,360,690,1020]
rows_y=[90,245,400,555]
row_data=[
 ('Provenance and fit','Independent trace / fit levels','Interpretation gap; verification','Paired effects + receipt trace'),
 ('Social capital','Tie type and route weighting','Correction uptake; remote reach','Topology comparison'),
 ('Capacity and fatigue','Channel allocation; thresholds','Accessible actions; enactment','Sensitivity profile'),
 ('Reply access','Authority and response gates','Blame; active-question focus','Qualitative case set + boundary report'),
]
nodes=[]
for ci,(prefix,label) in enumerate(cols):
 for ri,row in enumerate(row_data):
  nodes.append(dnode(f'{prefix}{ri}',xs[ci]+20,rows_y[ri],240,95,row[ci],f'Family {ri+1}', 'component' if ci<2 else ('data' if ci==2 else 'evidence'),'rect',label))
edges=[]
for ri,y in enumerate(rows_y):
 mid=y+47.5
 edges.extend([
  dedge(f'c{ri}',f'o{ri}',[(xs[0]+260,mid),(xs[1]+20,mid)],'flow'),
  dedge(f'o{ri}',f'm{ri}',[(xs[1]+260,mid),(xs[2]+20,mid)],'data'),
  dedge(f'm{ri}',f'x{ri}',[(xs[2]+260,mid),(xs[3]+20,mid)],'evidence'),
 ])
_html = diagram_html('Research design map','From construct to interpretable output','Four aligned research families preserve the chain from an explicitly defined construct through an authored operationalization and bounded measure to a reportable within-model output.',1320,700,nodes,edges,groups=[dgroup('gc',30,45,280,635,'Constructs'),dgroup('go',360,45,280,635,'Operationalizations'),dgroup('gm',690,45,280,635,'Measures'),dgroup('gx',1020,45,280,635,'Outputs')],legend=[('flow','Declared construct mapping'),('data','Measurement derivation'),('evidence','Reportable evidence')])
print("PASS: deterministic SVG diagram rendered; orthogonal routing and node separation validated.")
_html

PASS: deterministic SVG diagram rendered; orthogonal routing and node separation validated.


Research design map  From construct to interpretable output  Four aligned research families preserve the chain from an explicitly defined construct through an authored operationalization and bounded measure to a reportable within-model output.     From construct to interpretable output  Four aligned research families preserve the chain from an explicitly defined construct through an authored operationalization and bounded measure to a reportable within-model output.                Constructs     Operationalizations     Measures     Outputs                 CONSTRUCT    Provenance and fit    Family 1     CONSTRUCT    Social capital    Family 2     CONSTRUCT    Capacity and fatigue    Family 3     CONSTRUCT    Reply access    Family 4     OPERATIONALIZATION    Independent trace / fit levels    Family 1     OPERATIONALIZATION    Tie type and route weighting    Family 2     OPERATIONALIZATION    Channel allocation; thresholds    Family 3     OPERATIONALIZATION    Authority and response gates    Family 4     MEASURE    Interpretation gap;  verification    Family 1     MEASURE    Correction uptake; remote  reach    Family 2     MEASURE    Accessible actions; enactment    Family 3     MEASURE    Blame; active-question focus    Family 4     OUTPUT    Paired effects + receipt trace    Family 1     OUTPUT    Topology comparison    Family 2     OUTPUT    Sensitivity profile    Family 3     OUTPUT    Qualitative case set +  boundary report    Family 4         Declared construct mapping      Measurement derivation      Reportable evidence    Validated: 16 nodes · 12 edges · orthogonal routing · 0 crossings · 0 node incursions · 0 node overlaps   Text equivalent  Elements    Provenance and fit : Family 1   Social capital : Family 2   Capacity and fatigue : Family 3   Reply access : Family 4   Independent trace / fit levels : Family 1   Tie type and route weighting : Family 2   Channel allocation; thresholds : Family 3   Authority and response gates : Family 4   Interpretation gap; verification : Family 1   Correction uptake; remote reach : Family 2   Accessible actions; enactment : Family 3   Blame; active-question focus : Family 4   Paired effects + receipt trace : Family 1   Topology comparison : Family 2   Sensitivity profile : Family 3   Qualitative case set + boundary report : Family 4   Relationships   Provenance and fit → Independent trace / fit levels  Independent trace / fit levels → Interpretation gap; verification  Interpretation gap; verification → Paired effects + receipt trace  Social capital → Tie type and route weighting  Tie type and route weighting → Correction uptake; remote reach  Correction uptake; remote reach → Topology comparison  Capacity and fatigue → Channel allocation; thresholds  Channel allocation; thresholds → Accessible actions; enactment  Accessible actions; enactment → Sensitivity profile  Reply access → Authority and response gates  Authority and response gates → Blame; active-question focus  Blame; active-question focus → Qualitative case set + boundary report

### Validity and claims boundary

Nested evidentiary zones show what the simulation establishes internally, what requires an empirical bridge, and what is presently unsupported. No number of generated nights automatically moves a claim from the inner zone to a claim about real people or institutions.

In [4]:
nodes=[
 dnode('internal1',370,285,180,80,'Deterministic behavior','Same state and action produce the same receipts','system','rect','supported'),
 dnode('internal2',590,285,180,80,'Matched mechanism contrast','One authored condition changes one model outcome','system','rect','supported'),
 dnode('internal3',480,385,180,80,'Causal receipt tracing','Typed events explain divergence inside the model','evidence','rect','supported'),
 dnode('bridge1',210,165,200,80,'Ecological plausibility','Requires task comparison and domain review','boundary','rect','caution'),
 dnode('bridge2',500,145,200,80,'User interpretation','Requires human-subject study and analysis','boundary','rect','caution'),
 dnode('bridge3',790,165,200,80,'Intervention hypothesis','Requires external calibration and testing','boundary','rect','caution'),
 dnode('outside1',65,535,220,80,'Population prevalence','Not supplied by generated frequencies','risk','rect','not supported'),
 dnode('outside2',365,555,220,80,'Individual diagnosis / trust score','Not supported for real people','risk','rect','not supported'),
 dnode('outside3',665,555,220,80,'Real-world forecast','No calibrated predictive probability','risk','rect','not supported'),
 dnode('outside4',965,535,220,80,'Operational efficacy','No deployment claim without empirical validation','risk','rect','not supported'),
]
_html = diagram_html('Validity / claims-boundary diagram','Where CHORUS claims stop','Nested zones distinguish results established by the authored simulation, questions that require an empirical bridge, and uses that the present model does not support. The absence of arrows is deliberate: these are evidentiary jurisdictions, not automatic promotion steps.',1240,700,nodes,[],groups=[dgroup('outer',25,25,1190,650,'External world claims — independent empirical evidence required','boundary'),dgroup('bridge',150,95,940,420,'Empirical bridge — interpretive caution and validation','boundary'),dgroup('inner',320,245,560,250,'Within-model evidence — supported when verification passes','boundary')],notes=['No simulated sample size moves a claim outward across these boundaries.', 'Prediction, diagnosis, prevalence, and operational efficacy remain outside the supported zone.'])
print("PASS: deterministic SVG diagram rendered; orthogonal routing and node separation validated.")
_html

PASS: deterministic SVG diagram rendered; orthogonal routing and node separation validated.


Validity / claims-boundary diagram  Where CHORUS claims stop  Nested zones distinguish results established by the authored simulation, questions that require an empirical bridge, and uses that the present model does not support. The absence of arrows is deliberate: these are evidentiary jurisdictions, not automatic promotion steps.     Where CHORUS claims stop  Nested zones distinguish results established by the authored simulation, questions that require an empirical bridge, and uses that the present model does not support. The absence of arrows is deliberate: these are evidentiary jurisdictions, not automatic promotion steps.       External world claims — independent empirical evidence required     Empirical bridge — interpretive caution and validation     Within-model evidence — supported when verification passes     SUPPORTED    Deterministic  behavior    Same state and action  produce the same  receipts     SUPPORTED    Matched mechanism  contrast    One authored  condition changes one  model outcome     SUPPORTED    Causal receipt  tracing    Typed events explain  divergence inside the  model     CAUTION    Ecological plausibility    Requires task comparison  and domain review     CAUTION    User interpretation    Requires human-subject  study and analysis     CAUTION    Intervention hypothesis    Requires external  calibration and testing     NOT SUPPORTED    Population prevalence    Not supplied by generated  frequencies     NOT SUPPORTED    Individual diagnosis /  trust score    Not supported for real  people     NOT SUPPORTED    Real-world forecast    No calibrated predictive  probability     NOT SUPPORTED    Operational efficacy    No deployment claim without  empirical validation      No simulated sample size moves a claim outward across these boundaries.  Prediction, diagnosis, prevalence, and operational efficacy remain outside the supported zone.   Validated: 10 nodes · 0 edges · orthogonal routing · 0 crossings · 0 node incursions · 0 node overlaps   Text equivalent  Elements    Deterministic behavior : Same state and action produce the same receipts   Matched mechanism contrast : One authored condition changes one model outcome   Causal receipt tracing : Typed events explain divergence inside the model   Ecological plausibility : Requires task comparison and domain review   User interpretation : Requires human-subject study and analysis   Intervention hypothesis : Requires external calibration and testing   Population prevalence : Not supplied by generated frequencies   Individual diagnosis / trust score : Not supported for real people   Real-world forecast : No calibrated predictive probability   Operational efficacy : No deployment claim without empirical validation   Relationships   No connector relationships; the diagram uses nested evidentiary zones.

## Research-question matrix

Each question below is answerable with model variants. None, by itself, establishes a real-world causal effect.

In [5]:
research_questions = [
    ("RQ1", "Provenance × fit", "When source trace and local social fit vary independently, when does fluent framing outweigh attached context inside the model?", "Trace level; fit level", "Interpretation gap; perceived consensus; verification action", "Matched 3×3 factorial within seed"),
    ("RQ2", "Correction timing", "How does earlier or later source restoration change reach, blame concentration, and repair availability?", "Correction arrival minute", "Reach; blame; common ground; accessible choices", "Matched timing sweep"),
    ("RQ3", "Reply access", "How does unequal ability to answer a claim affect personalization of distributed failure?", "Target reply access; authority asymmetry", "Blame concentration; active-question focus", "Ablation plus authority-stratified comparison"),
    ("RQ4", "Fatigue composition", "Does the type of accumulated load alter enactment when discernment and total load are held constant?", "Allocation across five fatigue channels", "Choice availability; enactment; last-resort route", "Compositional matched variants"),
    ("RQ5", "Social capital", "How do trust and dependence routes alter spread and correction uptake?", "Tie type; tie strength proxy; source position", "Remote reach; correction propagation; consensus", "Topology and tie-type variants"),
    ("RQ6", "Shared register / divergent model", "When actors use the same register but hold different assumptions about evidence or responsibility, how often does apparent agreement conceal coordination failure?", "Code relation; world-model relation", "Common ground visible; active-question focus; coordination pressure", "2×2 communication design"),
    ("RQ7", "Ambient vs direct propagation", "How does treating a consequence as social atmosphere rather than content crossing change attribution and repair?", "Route layer", "Attribution accuracy within model; blame; provenance", "Compatibility-preserving route ablation"),
    ("RQ8", "Institutional delay", "How do authorization requirements and delayed official response interact with local corrective action?", "Approval threshold; delay; local authority", "Reach; verification capacity; repair timing", "Factorial institutional-access variant"),
    ("RQ9", "Non-amplification floor", "What harm is avoided when every beat retains a non-amplifying action, and what repair remains unreachable?", "Floor present/absent in research-only branch", "Avoided reach; unresolved gap; ethical violations", "Safety-constrained ablation; never player-facing without review"),
    ("RQ10", "Concurrent load", "How does the number and timing of simultaneous incidents affect follow-through independently of reading speed?", "Active room count; arrival density", "Fatigue channels; enactment; unfinished supports", "Clock-density sweep"),
    ("RQ11", "Cross-coalition translation", "Which translation routes reveal materially shared action despite conflicting coalition language?", "Surface code; repair move; audience model", "Common ground visible; direct repair uptake", "Matched language-surface variants"),
    ("RQ12", "Network repair", "Which sequence of distributed supporting actions makes a previously blocked ideal reachable?", "Support order; source rooms; time cost", "Choice-access transition; total harm; afterimage", "Path enumeration over support-building policies"),
]
_html = table_html("Within-model research questions", ("ID", "Mechanism", "Question", "Independent variables", "Outcomes", "Design"), research_questions, row_headers=True)
assert len(research_questions) == 12
print("Research matrix defines 12 mechanism-specific questions and explicit contrasts.")
_html

Research matrix defines 12 mechanism-specific questions and explicit contrasts.


ID,Mechanism,Question,Independent variables,Outcomes,Design
RQ1,Provenance × fit,"When source trace and local social fit vary independently, when does fluent framing outweigh attached context inside the model?",Trace level; fit level,Interpretation gap; perceived consensus; verification action,Matched 3×3 factorial within seed
RQ2,Correction timing,"How does earlier or later source restoration change reach, blame concentration, and repair availability?",Correction arrival minute,Reach; blame; common ground; accessible choices,Matched timing sweep
RQ3,Reply access,How does unequal ability to answer a claim affect personalization of distributed failure?,Target reply access; authority asymmetry,Blame concentration; active-question focus,Ablation plus authority-stratified comparison
RQ4,Fatigue composition,Does the type of accumulated load alter enactment when discernment and total load are held constant?,Allocation across five fatigue channels,Choice availability; enactment; last-resort route,Compositional matched variants
RQ5,Social capital,How do trust and dependence routes alter spread and correction uptake?,Tie type; tie strength proxy; source position,Remote reach; correction propagation; consensus,Topology and tie-type variants
RQ6,Shared register / divergent model,"When actors use the same register but hold different assumptions about evidence or responsibility, how often does apparent agreement conceal coordination failure?",Code relation; world-model relation,Common ground visible; active-question focus; coordination pressure,2×2 communication design
RQ7,Ambient vs direct propagation,How does treating a consequence as social atmosphere rather than content crossing change attribution and repair?,Route layer,Attribution accuracy within model; blame; provenance,Compatibility-preserving route ablation
RQ8,Institutional delay,How do authorization requirements and delayed official response interact with local corrective action?,Approval threshold; delay; local authority,Reach; verification capacity; repair timing,Factorial institutional-access variant
RQ9,Non-amplification floor,"What harm is avoided when every beat retains a non-amplifying action, and what repair remains unreachable?",Floor present/absent in research-only branch,Avoided reach; unresolved gap; ethical violations,Safety-constrained ablation; never player-facing without review
RQ10,Concurrent load,How does the number and timing of simultaneous incidents affect follow-through independently of reading speed?,Active room count; arrival density,Fatigue channels; enactment; unfinished supports,Clock-density sweep


In [6]:
question_families = {}
for _, family, *_ in research_questions:
    question_families[family] = question_families.get(family, 0) + 1
_html = cards_html("Research program coverage", [
    ("Questions", len(research_questions), "Each names a mechanism, independent variables, outcomes, and design."),
    ("Mechanism families", len(question_families), "No generic misinformation-literacy omnibus score."),
    ("Actor levels", 3, "Actor, relationship/room, and whole-house outcomes are represented."),
    ("Claim level", "Within-model", "External claims require a separate empirical protocol."),
])
print("Coverage check passed: questions span actor, relational, temporal, and network mechanisms.")
_html

Coverage check passed: questions span actor, relational, temporal, and network mechanisms.


Questions  12  Each names a mechanism, independent variables, outcomes, and design.    Mechanism families  12  No generic misinformation-literacy omnibus score.    Actor levels  3  Actor, relationship/room, and whole-house outcomes are represented.    Claim level  Within-model  External claims require a separate empirical protocol.

## Matched-variant architecture

The strongest default design changes one declared condition while preserving the seed, generated incident truth, actor identities, baseline relationships, arrival schedule, and choice policy wherever the research question permits.

In [7]:
variant_protocol = [
    (1, "Register", "Name the baseline implementation version, seed domain, policy, and outcome definitions."),
    (2, "Freeze", "Hold incident truth, actor generation, route IDs, and random draw stream constant."),
    (3, "Intervene", "Change one parameter, grammar rule, route, threshold, or schedule declared in advance."),
    (4, "Run", "Execute baseline and variant under identical deterministic policies and ordered events."),
    (5, "Verify", "Reject pairs that violate schema, coherence, bounds, or replay invariants."),
    (6, "Compare", "Compute paired differences and retain their full distribution rather than only the mean."),
    (7, "Trace", "Identify the earliest divergent event and follow its local and remote receipts."),
    (8, "Read", "Qualitatively inspect motivation, affect, relationship, epistemic access, and narrative continuity."),
    (9, "Stress", "Repeat across plausible parameter ranges, policies, and topology variants."),
    (10, "Report", "State finite coverage, exclusions, failures, sensitivity, and the model/world boundary."),
]
_html = table_html("Matched-variant protocol", ("Step", "Stage", "Requirement"), variant_protocol)
assert [step for step, _, _ in variant_protocol] == list(range(1, 11))
print("PASS: ten-step protocol connects controlled comparison to causal and qualitative inspection.")
_html

PASS: ten-step protocol connects controlled comparison to causal and qualitative inspection.


Step,Stage,Requirement
1,Register,"Name the baseline implementation version, seed domain, policy, and outcome definitions."
2,Freeze,"Hold incident truth, actor generation, route IDs, and random draw stream constant."
3,Intervene,"Change one parameter, grammar rule, route, threshold, or schedule declared in advance."
4,Run,Execute baseline and variant under identical deterministic policies and ordered events.
5,Verify,"Reject pairs that violate schema, coherence, bounds, or replay invariants."
6,Compare,Compute paired differences and retain their full distribution rather than only the mean.
7,Trace,Identify the earliest divergent event and follow its local and remote receipts.
8,Read,"Qualitatively inspect motivation, affect, relationship, epistemic access, and narrative continuity."
9,Stress,"Repeat across plausible parameter ranges, policies, and topology variants."
10,Report,"State finite coverage, exclusions, failures, sensitivity, and the model/world boundary."


### Experimental and comparative design

The matched-variant design freezes the shared world before changing one declared parameter, scenario rule, or actor-profile condition. Each variant is paired with a comparison rule that states when the contrast remains causally interpretable inside the model.

In [8]:
nodes=[
 dnode('baseline',40,280,220,110,'Baseline CHORUS','Declared version, seed domain, policy, and outcomes','system','rect','reference'),
 dnode('freeze',340,280,220,110,'Freeze matched world','Truth, actors, routes, schedule, and random stream','boundary','rect','control'),
 dnode('param',650,80,240,110,'Parameter variant','Change one threshold, magnitude, or timing','component','rect','intervention'),
 dnode('scenario',650,290,240,110,'Scenario variant','Change one grammar rule or mechanism assignment','component','rect','intervention'),
 dnode('actor',650,500,240,110,'Actor-profile variant','Change one role, repertoire, capacity, or tie condition','component','rect','intervention'),
 dnode('param-c',1020,80,270,110,'Valid comparison','Paired difference when all non-target conditions remain frozen','evidence','rect','analysis rule'),
 dnode('scenario-c',1020,290,270,110,'Valid comparison','Matched grammar contrast with incident truth and policy declared','evidence','rect','analysis rule'),
 dnode('actor-c',1020,500,270,110,'Valid comparison','Matched actor contrast without silently changing topology or evidence','evidence','rect','analysis rule'),
]
edges=[
 dedge('baseline','freeze',[(260,335),(340,335)],'flow','register'),
 dedge('freeze','param',[(560,305),(610,305),(610,135),(650,135)],'data','one change',(610,220)),
 dedge('freeze','scenario',[(560,335),(650,335)],'data','one change'),
 dedge('freeze','actor',[(560,365),(625,365),(625,555),(650,555)],'data','one change',(625,460)),
 dedge('param','param-c',[(890,135),(1020,135)],'evidence','paired'),
 dedge('scenario','scenario-c',[(890,345),(1020,345)],'evidence','paired'),
 dedge('actor','actor-c',[(890,555),(1020,555)],'evidence','paired'),
]
_html = diagram_html('Controlled comparative design diagram','Matched-variant comparison architecture','A valid CHORUS comparison freezes the shared world, changes one declared target, and applies a comparison rule appropriate to that target. Unmatched narrative substitutions are not treated as causal contrasts.',1340,680,nodes,edges,legend=[('flow','Study registration'),('data','Controlled intervention'),('evidence','Valid paired comparison')])
print("PASS: deterministic SVG diagram rendered; orthogonal routing and node separation validated.")
_html

PASS: deterministic SVG diagram rendered; orthogonal routing and node separation validated.


Controlled comparative design diagram  Matched-variant comparison architecture  A valid CHORUS comparison freezes the shared world, changes one declared target, and applies a comparison rule appropriate to that target. Unmatched narrative substitutions are not treated as causal contrasts.     Matched-variant comparison architecture  A valid CHORUS comparison freezes the shared world, changes one declared target, and applies a comparison rule appropriate to that target. Unmatched narrative substitutions are not treated as causal contrasts.                 register      one change      one change      one change      paired      paired      paired     REFERENCE    Baseline CHORUS    Declared version, seed  domain, policy, and  outcomes     CONTROL    Freeze matched world    Truth, actors, routes,  schedule, and random stream     INTERVENTION    Parameter variant    Change one threshold,  magnitude, or timing     INTERVENTION    Scenario variant    Change one grammar rule or  mechanism assignment     INTERVENTION    Actor-profile variant    Change one role, repertoire,  capacity, or tie condition     ANALYSIS RULE    Valid comparison    Paired difference when all  non-target conditions remain  frozen     ANALYSIS RULE    Valid comparison    Matched grammar contrast with  incident truth and policy declared     ANALYSIS RULE    Valid comparison    Matched actor contrast without  silently changing topology or  evidence         Study registration      Controlled intervention      Valid paired comparison    Validated: 8 nodes · 7 edges · orthogonal routing · 0 crossings · 0 node incursions · 0 node overlaps   Text equivalent  Elements    Baseline CHORUS : Declared version, seed domain, policy, and outcomes   Freeze matched world : Truth, actors, routes, schedule, and random stream   Parameter variant : Change one threshold, magnitude, or timing   Scenario variant : Change one grammar rule or mechanism assignment   Actor-profile variant : Change one role, repertoire, capacity, or tie condition   Valid comparison : Paired difference when all non-target conditions remain frozen   Valid comparison : Matched grammar contrast with incident truth and policy declared   Valid comparison : Matched actor contrast without silently changing topology or evidence   Relationships   Baseline CHORUS → Freeze matched world (register)  Freeze matched world → Parameter variant (one change)  Freeze matched world → Scenario variant (one change)  Freeze matched world → Actor-profile variant (one change)  Parameter variant → Valid comparison (paired)  Scenario variant → Valid comparison (paired)  Actor-profile variant → Valid comparison (paired)

## Example factorial scope

Large run counts are useful only after the factor space and interpretation are declared. This cell makes the arithmetic visible for a provenance-by-fit-by-correction-timing study.

In [9]:
factors = {
    "source_trace": (25, 55, 85),
    "social_fit": (25, 55, 85),
    "correction_timing": ("early", "mid", "late"),
    "policy": ("non-amplifying", "verification-first", "relationship-first", "randomized"),
}
seed_count = 256
cells = 1
for values in factors.values():
    cells *= len(values)
runs = cells * seed_count
paired_contrasts_per_seed = cells - 1
_html = cards_html("Illustrative preregistered factorial", [
    ("Factor cells", cells, "3 trace × 3 fit × 3 timing × 4 policy conditions."),
    ("Seeds", seed_count, "Contiguous or explicitly enumerated before execution."),
    ("Total runs", f"{runs:,}", "Synthetic runs, not human observations."),
    ("Baseline contrasts", f"{paired_contrasts_per_seed:,} / seed", "Every condition may be compared with one declared baseline."),
])
assert cells == 108 and runs == 27648
print("PASS: factorial arithmetic is explicit; sample size is not represented as population evidence.")
_html

PASS: factorial arithmetic is explicit; sample size is not represented as population evidence.


Factor cells  108  3 trace × 3 fit × 3 timing × 4 policy conditions.    Seeds  256  Contiguous or explicitly enumerated before execution.    Total runs  27,648  Synthetic runs, not human observations.    Baseline contrasts  107 / seed  Every condition may be compared with one declared baseline.

## Outcomes and measurement discipline

An outcome must be named at the correct level and interpreted according to its representation. Synthetic impression counts and bounded indices are useful for comparison, but they are not validated survey scales or naturally observed units.

In [10]:
outcomes = [
    ("Reach", "House / room", "Synthetic impressions and avoided impressions", "Count and paired difference", "Do not interpret as expected platform views."),
    ("Interpretation gap", "Room", "Distance between record and dominant authored reading", "Bounded index; trajectory; endpoint", "Not a psychometric belief score."),
    ("Blame concentration", "Room / target", "Personalization of distributed failure", "Bounded index; event-attributed change", "Not a diagnosis of scapegoating in a real group."),
    ("Common ground visible", "Relationship / house", "Legibility of materially shared action", "Bounded index; time to threshold", "Does not establish real agreement."),
    ("Active-question focus", "Conversation", "Persistence of the bounded question", "Bounded index; diversion events", "Does not classify every topic shift as evasion."),
    ("Perceived consensus", "Room / house", "Apparent social majority under repeated signals", "Bounded index; route contribution", "Not a survey estimate."),
    ("Discernment", "Actor", "Modeled recognition of the sound action", "Baseline and trajectory", "Not intelligence, media literacy, or moral worth."),
    ("Enactment", "Actor", "Modeled carrying power for available action", "Baseline, trajectory, access threshold", "Not a stable trait."),
    ("Fatigue channels", "Actor", "Five typed platform loads", "Composition and trajectory", "Not clinical assessment."),
    ("Choice access", "Actor / beat", "Structural and capacity availability", "Binary gate plus unmet requirements", "Unavailable does not mean unknowable or immoral."),
    ("Causal receipts", "Event", "Typed local and remote deltas", "Path and contribution analysis", "Internal provenance only."),
]
_html = table_html("Outcome register and interpretation boundary", ("Outcome", "Level", "Representation", "Summary", "Do not claim"), outcomes, row_headers=True)
print("Outcome register defines 11 measures and a misuse boundary for each.")
_html

Outcome register defines 11 measures and a misuse boundary for each.


Outcome,Level,Representation,Summary,Do not claim
Reach,House / room,Synthetic impressions and avoided impressions,Count and paired difference,Do not interpret as expected platform views.
Interpretation gap,Room,Distance between record and dominant authored reading,Bounded index; trajectory; endpoint,Not a psychometric belief score.
Blame concentration,Room / target,Personalization of distributed failure,Bounded index; event-attributed change,Not a diagnosis of scapegoating in a real group.
Common ground visible,Relationship / house,Legibility of materially shared action,Bounded index; time to threshold,Does not establish real agreement.
Active-question focus,Conversation,Persistence of the bounded question,Bounded index; diversion events,Does not classify every topic shift as evasion.
Perceived consensus,Room / house,Apparent social majority under repeated signals,Bounded index; route contribution,Not a survey estimate.
Discernment,Actor,Modeled recognition of the sound action,Baseline and trajectory,"Not intelligence, media literacy, or moral worth."
Enactment,Actor,Modeled carrying power for available action,"Baseline, trajectory, access threshold",Not a stable trait.
Fatigue channels,Actor,Five typed platform loads,Composition and trajectory,Not clinical assessment.
Choice access,Actor / beat,Structural and capacity availability,Binary gate plus unmet requirements,Unavailable does not mean unknowable or immoral.


### Measurement pipeline

Raw generated nights and ordered decision traces remain upstream of receipts and derived metrics. Quantitative and qualitative work stay in separate lanes until an explicit mixed-method synthesis reconciles them.

In [11]:
nodes=[
 dnode('nights',30,310,170,90,'Generated nights','Declared seeds and model variant','data','document','input'),
 dnode('traces',230,310,170,90,'Decision traces','Ordered choices, times, and access states','evidence','document','record'),
 dnode('receipts',430,310,170,90,'Effect receipts','Local, remote, pulse, and afterimage provenance','evidence','document','record'),
 dnode('metrics',630,310,170,90,'Derived metrics','Paired differences, distributions, transitions','data','rect','derivation'),
 dnode('quant',850,140,180,90,'Quantitative analysis','Effect distributions, uncertainty, sensitivity','component','rect','analysis lane'),
 dnode('qual',850,480,180,90,'Qualitative coding','Motivation, affect, relations, narrative continuity','component','rect','analysis lane'),
 dnode('synth',1090,300,190,120,'Mixed-method synthesis','Convergence, divergence, mechanism cases, model revision','evidence','rect','integration'),
 dnode('report',1330,315,180,90,'Boundary-aware report','Finite domain, failures, sensitivity, claim limits','evidence','document','output'),
]
edges=[
 dedge('nights','traces',[(200,355),(230,355)],'data'),
 dedge('traces','receipts',[(400,355),(430,355)],'evidence'),
 dedge('receipts','metrics',[(600,355),(630,355)],'data'),
 dedge('metrics','quant',[(800,335),(820,335),(820,185),(850,185)],'data','numeric'),
 dedge('metrics','qual',[(800,375),(830,375),(830,525),(850,525)],'evidence','cases'),
 dedge('quant','synth',[(1030,185),(1060,185),(1060,340),(1090,340)],'evidence'),
 dedge('qual','synth',[(1030,525),(1070,525),(1070,380),(1090,380)],'evidence'),
 dedge('synth','report',[(1280,360),(1330,360)],'evidence','report'),
]
_html = diagram_html('Measurement and analysis pipeline','From generated run to mixed-method finding','The pipeline preserves raw event provenance before deriving metrics, separates quantitative and qualitative analysis, and reunites them only in an explicit mixed-method synthesis with claim boundaries.',1540,680,nodes,edges,legend=[('data','Generated or derived data'),('evidence','Retained provenance / analytic evidence')])
print("PASS: deterministic SVG diagram rendered; orthogonal routing and node separation validated.")
_html

PASS: deterministic SVG diagram rendered; orthogonal routing and node separation validated.


Measurement and analysis pipeline  From generated run to mixed-method finding  The pipeline preserves raw event provenance before deriving metrics, separates quantitative and qualitative analysis, and reunites them only in an explicit mixed-method synthesis with claim boundaries.     From generated run to mixed-method finding  The pipeline preserves raw event provenance before deriving metrics, separates quantitative and qualitative analysis, and reunites them only in an explicit mixed-method synthesis with claim boundaries.                 numeric      cases        report     INPUT    Generated nights    Declared seeds and  model variant     RECORD    Decision traces    Ordered choices,  times, and access  states     RECORD    Effect receipts    Local, remote,  pulse, and  afterimage  provenance     DERIVATION    Derived metrics    Paired differences,  distributions,  transitions     ANALYSIS LANE    Quantitative analysis    Effect distributions,  uncertainty,  sensitivity     ANALYSIS LANE    Qualitative coding    Motivation, affect,  relations, narrative  continuity     INTEGRATION    Mixed-method synthesis    Convergence,  divergence, mechanism  cases, model revision     OUTPUT    Boundary-aware report    Finite domain,  failures,  sensitivity, claim  limits         Generated or derived data      Retained provenance / analytic evidence    Validated: 8 nodes · 8 edges · orthogonal routing · 0 crossings · 0 node incursions · 0 node overlaps   Text equivalent  Elements    Generated nights : Declared seeds and model variant   Decision traces : Ordered choices, times, and access states   Effect receipts : Local, remote, pulse, and afterimage provenance   Derived metrics : Paired differences, distributions, transitions   Quantitative analysis : Effect distributions, uncertainty, sensitivity   Qualitative coding : Motivation, affect, relations, narrative continuity   Mixed-method synthesis : Convergence, divergence, mechanism cases, model revision   Boundary-aware report : Finite domain, failures, sensitivity, claim limits   Relationships   Generated nights → Decision traces  Decision traces → Effect receipts  Effect receipts → Derived metrics  Derived metrics → Quantitative analysis (numeric)  Derived metrics → Qualitative coding (cases)  Quantitative analysis → Mixed-method synthesis  Qualitative coding → Mixed-method synthesis  Mixed-method synthesis → Boundary-aware report (report)

## Quantitative analysis plan

The preferred analysis treats seeds as matched generated worlds, policies as declared decision rules, and variant conditions as interventions on the model.

In [12]:
quantitative = [
    ("Integrity", "Count rejected packs, invariant failures, replay failures, exhausted choice paths, and non-finite values before outcome analysis."),
    ("Paired effects", "For each seed-policy pair, compute variant minus baseline at predeclared endpoints and over trajectories."),
    ("Distributions", "Report median, interquartile range, tails, sign consistency, and complete min/max alongside means."),
    ("Event timing", "Compare time to correction, first direct crossing, first blocked ideal, room close, and whole-night completion."),
    ("Path contribution", "Attribute outcome changes to typed receipts and identify earliest divergence."),
    ("Interaction", "Estimate whether factor combinations change direction or magnitude beyond their separate within-model effects."),
    ("Sensitivity", "Repeat under alternative thresholds, effect magnitudes, policies, topologies, and grammar ablations."),
    ("Multiplicity", "Separate confirmatory questions from exploratory screens and control or clearly label repeated comparisons."),
    ("Missingness", "Treat rejected or incoherent runs as substantive model failures; do not silently drop them."),
]
_html = checklist_html("Quantitative analysis commitments", [("REQUIRED", label, procedure) for label, procedure in quantitative])
print("Quantitative plan contains nine commitments from integrity screening through missingness.")
_html

Quantitative plan contains nine commitments from integrity screening through missingness.


REQUIRED  Integrity  Count rejected packs, invariant failures, replay failures, exhausted choice paths, and non-finite values before outcome analysis.    REQUIRED  Paired effects  For each seed-policy pair, compute variant minus baseline at predeclared endpoints and over trajectories.    REQUIRED  Distributions  Report median, interquartile range, tails, sign consistency, and complete min/max alongside means.    REQUIRED  Event timing  Compare time to correction, first direct crossing, first blocked ideal, room close, and whole-night completion.    REQUIRED  Path contribution  Attribute outcome changes to typed receipts and identify earliest divergence.    REQUIRED  Interaction  Estimate whether factor combinations change direction or magnitude beyond their separate within-model effects.    REQUIRED  Sensitivity  Repeat under alternative thresholds, effect magnitudes, policies, topologies, and grammar ablations.    REQUIRED  Multiplicity  Separate confirmatory questions from exploratory screens and control or clearly label repeated comparisons.    REQUIRED  Missingness  Treat rejected or incoherent runs as substantive model failures; do not silently drop them.

In [13]:
tidy_schema = [
    ("run_id", "string", "Implementation version + seed + policy + variant"),
    ("seed", "integer", "Generated-world pairing key"),
    ("variant_id", "string", "Predeclared model intervention"),
    ("policy_id", "string", "Deterministic or randomized choice policy with its own seed"),
    ("event_index", "integer", "Ordered causal record position"),
    ("logical_minute", "integer", "House time; never wall-clock reading time"),
    ("source_room", "string", "Originating room"),
    ("target_room", "string", "Receiving room for an effect receipt"),
    ("event_kind", "enum", "Choice or autonomous pulse"),
    ("effect_layer", "enum", "Local, ambient remote, or compatible direct crossing"),
    ("metric", "string", "One declared outcome or state variable"),
    ("value_before", "number", "Finite bounded or count value"),
    ("delta", "number", "Typed receipt contribution"),
    ("value_after", "number", "Post-transition value"),
    ("coherence_status", "enum", "Pass, rejected, or review-required"),
]
_html = table_html("Tidy synthetic event schema", ("Field", "Type", "Meaning"), tidy_schema, row_headers=True)
assert len(tidy_schema) == 15
print("Synthetic data schema retains pairing, event order, route provenance, and coherence status.")
_html

Synthetic data schema retains pairing, event order, route provenance, and coherence status.


Field,Type,Meaning
run_id,string,Implementation version + seed + policy + variant
seed,integer,Generated-world pairing key
variant_id,string,Predeclared model intervention
policy_id,string,Deterministic or randomized choice policy with its own seed
event_index,integer,Ordered causal record position
logical_minute,integer,House time; never wall-clock reading time
source_room,string,Originating room
target_room,string,Receiving room for an effect receipt
event_kind,enum,Choice or autonomous pulse
effect_layer,enum,"Local, ambient remote, or compatible direct crossing"


## Qualitative analysis plan

A numerical pass cannot determine whether a generated person remains specific, motivated, affectively continuous, relationally situated, and epistemically bounded. Close reading is therefore a release and research method, not decorative interpretation.

In [14]:
qualitative_codes = [
    ("Actor specificity", "The seat has concrete role, objective, stakes, resources, relationships, and conflicts rather than a demographic or ideological placeholder."),
    ("Motivational continuity", "Each choice remains traceable to represented goals, incentives, protection, authority, or capacity."),
    ("Affective continuity", "Pressure and recovery follow events; feeling does not appear solely to force a plot turn."),
    ("Relational coherence", "Care, trust, dependence, competition, authority, and accountability remain consistent unless an event changes them."),
    ("Epistemic discipline", "The seat acts only on available record, interpretation, and explicit uncertainty."),
    ("Communication situatedness", "Register choice fits audience and context without defining the actor's essence or belief."),
    ("Narrative causality", "Later beats and afterimages preserve consequences rather than resetting for a new lesson."),
    ("Cross-room plausibility", "Remote effects are appropriate to the route and do not invent unsupported shared content."),
    ("Ethical legibility", "Responsibility remains visible without diagnostic certainty, identity inference, or coerced amplification."),
    ("Counterexample value", "The case challenges rather than merely confirms the expected mechanism."),
]
_html = table_html("Qualitative coherence codebook", ("Code", "Review question"), qualitative_codes, row_headers=True)
print("Qualitative codebook defines ten review dimensions for generated nights and matched variants.")
_html

Qualitative codebook defines ten review dimensions for generated nights and matched variants.


Code,Review question
Actor specificity,"The seat has concrete role, objective, stakes, resources, relationships, and conflicts rather than a demographic or ideological placeholder."
Motivational continuity,"Each choice remains traceable to represented goals, incentives, protection, authority, or capacity."
Affective continuity,Pressure and recovery follow events; feeling does not appear solely to force a plot turn.
Relational coherence,"Care, trust, dependence, competition, authority, and accountability remain consistent unless an event changes them."
Epistemic discipline,"The seat acts only on available record, interpretation, and explicit uncertainty."
Communication situatedness,Register choice fits audience and context without defining the actor's essence or belief.
Narrative causality,Later beats and afterimages preserve consequences rather than resetting for a new lesson.
Cross-room plausibility,Remote effects are appropriate to the route and do not invent unsupported shared content.
Ethical legibility,"Responsibility remains visible without diagnostic certainty, identity inference, or coerced amplification."
Counterexample value,The case challenges rather than merely confirms the expected mechanism.


In [15]:
rating_scale = [
    (0, "Contradicted", "The generated evidence directly violates the criterion."),
    (1, "Weak", "The criterion is nominally present but generic, under-motivated, or discontinuous."),
    (2, "Adequate", "The criterion is coherent enough for use, with a documented limitation."),
    (3, "Strong", "The criterion is specific, continuous, and supported across the relevant record."),
    ("R", "Review required", "Evidence is ambiguous, ethically sensitive, or cannot be judged from the available record."),
]
_html = table_html("Ordinal qualitative review scale", ("Code", "Label", "Decision rule"), rating_scale)
print("Review scale preserves an explicit uncertainty category instead of forcing false precision.")
_html

Review scale preserves an explicit uncertainty category instead of forcing false precision.


Code,Label,Decision rule
0,Contradicted,The generated evidence directly violates the criterion.
1,Weak,"The criterion is nominally present but generic, under-motivated, or discontinuous."
2,Adequate,"The criterion is coherent enough for use, with a documented limitation."
3,Strong,"The criterion is specific, continuous, and supported across the relevant record."
R,Review required,"Evidence is ambiguous, ethically sensitive, or cannot be judged from the available record."


## Mixed-method integration

Quantitative and qualitative evidence answer different failure modes. They should meet at the level of a specific generated pair, event path, or mechanism—not in a generic narrative added after the statistics.

In [16]:
integration = [
    ("Convergent", "Distribution shifts in the predicted direction and reviewed cases preserve the proposed mechanism.", "Report effect, cases, sensitivity, and limits."),
    ("Quantitative-only", "A stable numerical difference lacks plausible or coherent case-level mechanism.", "Treat as a model artifact until traced and repaired."),
    ("Qualitative-only", "Compelling cases exist but aggregate direction is unstable or rare.", "Narrow the claim to a boundary condition or case class."),
    ("Divergent", "Numbers and close reading support opposing interpretations.", "Pause conclusion; inspect metric definition, coding, thresholds, and hidden coupling."),
    ("Null", "No meaningful paired difference and no case-level mechanism appears.", "Retain the null result; do not search post hoc for a new outcome."),
    ("Incoherent", "Variant causes rejected packs, broken motives, impossible knowledge, or route violations.", "The intervention is not a valid test in its current form."),
]
_html = table_html("Mixed-method joint display", ("Pattern", "Finding", "Required response"), integration, row_headers=True)
print("Joint display defines six result patterns, including null and incoherent outcomes.")
_html

Joint display defines six result patterns, including null and incoherent outcomes.


Pattern,Finding,Required response
Convergent,Distribution shifts in the predicted direction and reviewed cases preserve the proposed mechanism.,"Report effect, cases, sensitivity, and limits."
Quantitative-only,A stable numerical difference lacks plausible or coherent case-level mechanism.,Treat as a model artifact until traced and repaired.
Qualitative-only,Compelling cases exist but aggregate direction is unstable or rare.,Narrow the claim to a boundary condition or case class.
Divergent,Numbers and close reading support opposing interpretations.,"Pause conclusion; inspect metric definition, coding, thresholds, and hidden coupling."
Null,No meaningful paired difference and no case-level mechanism appears.,Retain the null result; do not search post hoc for a new outcome.
Incoherent,"Variant causes rejected packs, broken motives, impossible knowledge, or route violations.",The intervention is not a valid test in its current form.


## Robustness and falsification

A research program becomes scholarly when it states what could change its mind. CHORUS claims should be narrowed, revised, or rejected when their mechanism fails under declared checks.

In [17]:
falsification = [
    ("Mechanism absence", "The expected outcome difference appears without the proposed event path.", "Reject the causal explanation even if the endpoint difference remains."),
    ("Sign instability", "Reasonable policies, seeds, thresholds, or topologies reverse the effect.", "Report a conditional result or withdraw the general within-model claim."),
    ("Construct collapse", "Two variables intended to be distinct always move together by implementation.", "Refactor or narrow the construct claim."),
    ("Narrative incoherence", "The variant requires actors to violate motives, relationships, knowledge, or continuity.", "Reject the variant as an invalid operationalization."),
    ("Ethical failure", "The study design encourages identity inference, diagnosis, coerced amplification, or responsibility transfer.", "Stop the study path regardless of statistical clarity."),
    ("External contradiction", "Empirical evidence conflicts with the model's assumed mechanism.", "Revise assumptions; do not defend the simulation as self-validating."),
    ("No discriminant behavior", "Changing a construct produces no distinguishable consequence where the model says it matters.", "Question whether the variable has functional meaning."),
]
_html = table_html("Evidence that narrows or rejects a claim", ("Failure", "Observation", "Decision"), falsification, row_headers=True)
print("Falsification register names seven conditions that can overturn a model claim.")
_html

Falsification register names seven conditions that can overturn a model claim.


Failure,Observation,Decision
Mechanism absence,The expected outcome difference appears without the proposed event path.,Reject the causal explanation even if the endpoint difference remains.
Sign instability,"Reasonable policies, seeds, thresholds, or topologies reverse the effect.",Report a conditional result or withdraw the general within-model claim.
Construct collapse,Two variables intended to be distinct always move together by implementation.,Refactor or narrow the construct claim.
Narrative incoherence,"The variant requires actors to violate motives, relationships, knowledge, or continuity.",Reject the variant as an invalid operationalization.
Ethical failure,"The study design encourages identity inference, diagnosis, coerced amplification, or responsibility transfer.",Stop the study path regardless of statistical clarity.
External contradiction,Empirical evidence conflicts with the model's assumed mechanism.,Revise assumptions; do not defend the simulation as self-validating.
No discriminant behavior,Changing a construct produces no distinguishable consequence where the model says it matters.,Question whether the variable has functional meaning.


In [18]:
robustness = [
    ("Seed domain", "Repeat on a disjoint preregistered domain and report overlap and divergence."),
    ("Policy", "Use at least one contrasting deterministic policy and one randomized policy with recorded randomness."),
    ("Threshold", "Sweep access and propagation thresholds around every reported discontinuity."),
    ("Magnitude", "Scale effect deltas while preserving sign, then test sign alternatives where theoretically plausible."),
    ("Topology", "Test route removal, density, and centrality alternatives."),
    ("Grammar", "Repeat after actor-role and communication-dynamic ablation."),
    ("Coding", "Use independent qualitative review and reconcile disagreements against the event record."),
    ("Metric", "Check whether conclusions persist under adjacent outcomes rather than one favored index."),
]
_html = checklist_html("Robustness suite", [("STRESS", label, procedure) for label, procedure in robustness])
print("Robustness suite covers eight independent sources of model sensitivity.")
_html

Robustness suite covers eight independent sources of model sensitivity.


STRESS  Seed domain  Repeat on a disjoint preregistered domain and report overlap and divergence.    STRESS  Policy  Use at least one contrasting deterministic policy and one randomized policy with recorded randomness.    STRESS  Threshold  Sweep access and propagation thresholds around every reported discontinuity.    STRESS  Magnitude  Scale effect deltas while preserving sign, then test sign alternatives where theoretically plausible.    STRESS  Topology  Test route removal, density, and centrality alternatives.    STRESS  Grammar  Repeat after actor-role and communication-dynamic ablation.    STRESS  Coding  Use independent qualitative review and reconcile disagreements against the event record.    STRESS  Metric  Check whether conclusions persist under adjacent outcomes rather than one favored index.

## Ethics and data governance

The current simulation generates fictional state locally and requires no telemetry. A study of model outputs is different from a study involving people. The boundary changes as soon as researchers recruit participants, collect interaction traces, link identity, or evaluate real-world claims.

In [19]:
ethics = [
    ("Synthetic model audit", "Generated packs, event logs, and code only", "No human-subject claim; still review misuse, bias, identity inference, and youth safety."),
    ("Usability study", "Participant observation or feedback", "Consent, accessible participation, data minimization, withdrawal, and institutional review determination."),
    ("Learning study", "Pre/post measures or comparison groups", "Validated measures, power rationale, preregistration, privacy, debriefing, and review."),
    ("Behavioral telemetry", "Fine-grained interaction traces", "Not collected by the product; any research build needs separate consent, minimization, retention, and security."),
    ("Real-world case mapping", "External people, communities, or incidents", "High risk of diagnosis, defamation, reidentification, and false attribution; requires independent evidence and ethical/legal review."),
    ("Youth participation", "Minors or youth-directed evaluation", "Heightened consent/assent, safeguarding, content, privacy, and power review."),
]
_html = table_html("Research-ethics boundary by study type", ("Study type", "Data", "Additional requirement"), ethics, row_headers=True)
print("Ethics table separates synthetic audit from five increasingly sensitive empirical designs.")
_html

Ethics table separates synthetic audit from five increasingly sensitive empirical designs.


Study type,Data,Additional requirement
Synthetic model audit,"Generated packs, event logs, and code only","No human-subject claim; still review misuse, bias, identity inference, and youth safety."
Usability study,Participant observation or feedback,"Consent, accessible participation, data minimization, withdrawal, and institutional review determination."
Learning study,Pre/post measures or comparison groups,"Validated measures, power rationale, preregistration, privacy, debriefing, and review."
Behavioral telemetry,Fine-grained interaction traces,"Not collected by the product; any research build needs separate consent, minimization, retention, and security."
Real-world case mapping,"External people, communities, or incidents","High risk of diagnosis, defamation, reidentification, and false attribution; requires independent evidence and ethical/legal review."
Youth participation,Minors or youth-directed evaluation,"Heightened consent/assent, safeguarding, content, privacy, and power review."


In [20]:
governance = [
    ("Collect nothing by default", "The production application remains local and telemetry-free."),
    ("Separate research build", "Any participant logging must be technically and visibly distinct from ordinary play."),
    ("Purpose limitation", "Collect only fields necessary for the preregistered question."),
    ("No covert inference", "Do not infer ideology, diagnosis, deception, or identity from play style."),
    ("Pseudonymize early", "Separate contact, consent, and study data; minimize linkage."),
    ("Bound retention", "Declare deletion dates, access roles, backups, and export handling."),
    ("Report exclusions", "Document withdrawals, failed runs, accessibility barriers, and missing data."),
    ("Publish synthetic examples", "Prefer generated cases and aggregate data over participant narratives."),
]
_html = checklist_html("Data-governance commitments", [("REQUIRED", label, procedure) for label, procedure in governance])
print("Data governance preserves the product's privacy baseline and forbids covert profiling.")
_html

Data governance preserves the product's privacy baseline and forbids covert profiling.


REQUIRED  Collect nothing by default  The production application remains local and telemetry-free.    REQUIRED  Separate research build  Any participant logging must be technically and visibly distinct from ordinary play.    REQUIRED  Purpose limitation  Collect only fields necessary for the preregistered question.    REQUIRED  No covert inference  Do not infer ideology, diagnosis, deception, or identity from play style.    REQUIRED  Pseudonymize early  Separate contact, consent, and study data; minimize linkage.    REQUIRED  Bound retention  Declare deletion dates, access roles, backups, and export handling.    REQUIRED  Report exclusions  Document withdrawals, failed runs, accessibility barriers, and missing data.    REQUIRED  Publish synthetic examples  Prefer generated cases and aggregate data over participant narratives.

### Ethics and risk controls

Ethical controls are represented as gates rather than a general disclaimer. The final branch distinguishes explanatory and educational research from operational targeting, individual scoring, or coercive prediction.

In [21]:
nodes=[
 dnode('content',40,280,190,100,'Fictional content boundary','PG-safe composites; no operational target dossier','boundary','rect','gate 1'),
 dnode('overclaim',290,280,190,100,'Anti-overclaim controls','Synthetic indices; declared assumptions and limits','boundary','rect','gate 2'),
 dnode('certainty',540,280,190,100,'Predictive-certainty ban','No diagnosis, trust score, or calibrated forecast','boundary','rect','gate 3'),
 dnode('use',790,270,190,120,'Use boundary','Is the proposed use explanatory / educational?','decision','diamond','decision gate'),
 dnode('allowed',1050,100,230,110,'Allowed use','Mechanism study, teaching, design critique, research hypothesis','system','rect','educational / research'),
 dnode('blocked',1050,470,230,110,'Blocked use','Operational targeting, individual scoring, coercive prediction','risk','rect','prohibited'),
]
edges=[
 dedge('content','overclaim',[(230,330),(290,330)],'boundary','passes'),
 dedge('overclaim','certainty',[(480,330),(540,330)],'boundary','passes'),
 dedge('certainty','use',[(730,330),(790,330)],'boundary','review'),
 dedge('use','allowed',[(980,300),(1010,300),(1010,155),(1050,155)],'flow','yes',(1010,230)),
 dedge('use','blocked',[(980,360),(1020,360),(1020,525),(1050,525)],'boundary','no',(1020,445)),
]
_html = diagram_html('Ethics and misuse-control diagram','Ethics and risk-control gates','A proposed use must pass content, interpretation, and predictive-certainty controls before reaching the educational/research branch. The blocked branch records prohibited uses rather than offering an alternative operational workflow.',1330,660,nodes,edges,legend=[('boundary','Guardrail / prohibition boundary'),('flow','Permitted explanatory use')])
print("PASS: deterministic SVG diagram rendered; orthogonal routing and node separation validated.")
_html

PASS: deterministic SVG diagram rendered; orthogonal routing and node separation validated.


Ethics and misuse-control diagram  Ethics and risk-control gates  A proposed use must pass content, interpretation, and predictive-certainty controls before reaching the educational/research branch. The blocked branch records prohibited uses rather than offering an alternative operational workflow.     Ethics and risk-control gates  A proposed use must pass content, interpretation, and predictive-certainty controls before reaching the educational/research branch. The blocked branch records prohibited uses rather than offering an alternative operational workflow.              passes      passes      review      yes      no     GATE 1    Fictional content  boundary    PG-safe composites; no  operational target  dossier     GATE 2    Anti-overclaim controls    Synthetic indices;  declared assumptions  and limits     GATE 3    Predictive-certainty  ban    No diagnosis, trust  score, or calibrated  forecast     DECISION GATE    Use boundary    Is the proposed use  explanatory /  educational?     EDUCATIONAL / RESEARCH    Allowed use    Mechanism study, teaching,  design critique, research  hypothesis     PROHIBITED    Blocked use    Operational targeting,  individual scoring, coercive  prediction         Guardrail / prohibition boundary      Permitted explanatory use    Validated: 6 nodes · 5 edges · orthogonal routing · 0 crossings · 0 node incursions · 0 node overlaps   Text equivalent  Elements    Fictional content boundary : PG-safe composites; no operational target dossier   Anti-overclaim controls : Synthetic indices; declared assumptions and limits   Predictive-certainty ban : No diagnosis, trust score, or calibrated forecast   Use boundary : Is the proposed use explanatory / educational?   Allowed use : Mechanism study, teaching, design critique, research hypothesis   Blocked use : Operational targeting, individual scoring, coercive prediction   Relationships   Fictional content boundary → Anti-overclaim controls (passes)  Anti-overclaim controls → Predictive-certainty ban (passes)  Predictive-certainty ban → Use boundary (review)  Use boundary → Allowed use (yes)  Use boundary → Blocked use (no)

## Reporting template

A complete report should let another reader reconstruct the question, intervention, run domain, exclusions, mechanism trace, qualitative judgment, and claim boundary.

In [22]:
report_sections = [
    (1, "Question and claim level", "State whether the claim is structural, within-model comparative, exploratory, or empirical."),
    (2, "Model version", "Record commit or release binding, generator version, notebook build, and changed assumptions."),
    (3, "Design", "Declare factors, conditions, seed domain, policies, controls, outcomes, and stopping rule."),
    (4, "Integrity results", "Report generation rejection, invariant, replay, and coherence-review results before outcomes."),
    (5, "Quantitative results", "Present paired distributions, uncertainty, interactions, tails, and all preregistered outcomes."),
    (6, "Mechanism trace", "Show representative causal paths and the earliest divergence."),
    (7, "Qualitative results", "Report coding, disagreement, counterexamples, and narrative/ethical failures."),
    (8, "Sensitivity", "Show threshold, policy, topology, magnitude, grammar, and metric robustness."),
    (9, "Limitations", "Name synthetic, construct, coverage, ecological, and external-validity limits."),
    (10, "Decision", "State what was supported, narrowed, rejected, or left unresolved."),
    (11, "Reproduction", "Provide code, configs, seed list, policy definitions, raw synthetic results, and environment."),
]
_html = table_html("Minimum study-report structure", ("Order", "Section", "Required contents"), report_sections)
assert [row[0] for row in report_sections] == list(range(1, 12))
print("Reporting contract contains 11 ordered sections from claim level through reproduction.")
_html

Reporting contract contains 11 ordered sections from claim level through reproduction.


Order,Section,Required contents
1,Question and claim level,"State whether the claim is structural, within-model comparative, exploratory, or empirical."
2,Model version,"Record commit or release binding, generator version, notebook build, and changed assumptions."
3,Design,"Declare factors, conditions, seed domain, policies, controls, outcomes, and stopping rule."
4,Integrity results,"Report generation rejection, invariant, replay, and coherence-review results before outcomes."
5,Quantitative results,"Present paired distributions, uncertainty, interactions, tails, and all preregistered outcomes."
6,Mechanism trace,Show representative causal paths and the earliest divergence.
7,Qualitative results,"Report coding, disagreement, counterexamples, and narrative/ethical failures."
8,Sensitivity,"Show threshold, policy, topology, magnitude, grammar, and metric robustness."
9,Limitations,"Name synthetic, construct, coverage, ecological, and external-validity limits."
10,Decision,"State what was supported, narrowed, rejected, or left unresolved."


## Research roadmap

The highest-value sequence begins with internal construct and mechanism validity, then moves to human evaluation only where a bounded question and ethical protocol justify it.

In [23]:
roadmap = [
    ("1 · Model audit", "Variable definitions, separations, coherence gates, source ownership", "Complete specification and contradiction log"),
    ("2 · Sensitivity", "Matched seeds across thresholds, policies, topology, and grammar", "Mechanism robustness map"),
    ("3 · Qualitative review", "Independent coding of generated actors, motives, affect, relationships, and epistemic boundaries", "Inter-rater record and model revisions"),
    ("4 · Expert review", "Information behavior, social science, ethics, game studies, accessibility, and domain critique", "Content-validity and misuse findings"),
    ("5 · Usability / comprehension", "Can readers distinguish record, inference, motive, unknown, and claim limits?", "Accessible participant protocol and bounded findings"),
    ("6 · Comparative learning study", "Only after measures and intervention are externally justified", "Empirical evidence that remains separate from simulation output"),
]
_html = table_html("Staged research roadmap", ("Stage", "Question", "Deliverable"), roadmap, row_headers=True)
print("Roadmap places model audit and sensitivity before participant-facing efficacy claims.")
_html

Roadmap places model audit and sensitivity before participant-facing efficacy claims.


Stage,Question,Deliverable
1 · Model audit,"Variable definitions, separations, coherence gates, source ownership",Complete specification and contradiction log
2 · Sensitivity,"Matched seeds across thresholds, policies, topology, and grammar",Mechanism robustness map
3 · Qualitative review,"Independent coding of generated actors, motives, affect, relationships, and epistemic boundaries",Inter-rater record and model revisions
4 · Expert review,"Information behavior, social science, ethics, game studies, accessibility, and domain critique",Content-validity and misuse findings
5 · Usability / comprehension,"Can readers distinguish record, inference, motive, unknown, and claim limits?",Accessible participant protocol and bounded findings
6 · Comparative learning study,Only after measures and intervention are externally justified,Empirical evidence that remains separate from simulation output


## Research-design conclusion

CHORUS can support rigorous research practice when every result remains attached to its model version, assumptions, seed domain, policy, causal receipts, coherence review, sensitivity, and claim boundary.

Its most defensible immediate contribution is not prediction. It is a reproducible way to formulate and inspect questions about distributed sensemaking, social pressure, information propagation, relational constraints, and the gap between recognizing a sound action and being able to carry it.